# LoanSense — Credit Risk Model Training

**Dataset:** German Credit Dataset (UCI Machine Learning Repository)  
**Goal:** Predict probability of loan default (binary classification)  
**Models compared:** Logistic Regression · Random Forest · XGBoost

---

## 1. Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

from urllib.request import urlretrieve
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, classification_report, confusion_matrix
)
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from xgboost import XGBClassifier, plot_importance

plt.style.use('dark_background')
sns.set_theme(style='darkgrid')
print('Setup complete.')

## 2. Load Dataset

In [ ]:
# Download German Credit Dataset from UCI
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/german/german.data'
urlretrieve(url, 'german.data')

col_names = [
    'checking_account', 'duration', 'credit_history', 'purpose',
    'credit_amount', 'savings', 'employment', 'installment_rate',
    'personal_status', 'other_debtors', 'residence_since',
    'property', 'age', 'other_installments', 'housing',
    'existing_credits', 'job', 'num_dependents', 'telephone',
    'foreign_worker', 'target'
]

df = pd.read_csv('german.data', sep=' ', header=None, names=col_names)
# 1 = good credit, 2 = bad credit → remap to 0/1
df['target'] = (df['target'] == 2).astype(int)

print(f'Shape: {df.shape}')
print(f'Default rate: {df["target"].mean():.1%}')
df.head()

## 3. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Class distribution
counts = df['target'].value_counts()
axes[0].bar(['Good Credit (0)', 'Default (1)'], counts.values, color=['#22c55e', '#ef4444'])
axes[0].set_title('Class Distribution')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 5, str(v), ha='center', fontweight='bold')

# Credit amount distribution by class
df.groupby('target')['credit_amount'].plot(kind='density', ax=axes[1],
    label=['Good', 'Default'], color=['#22c55e', '#ef4444'])
axes[1].set_title('Credit Amount Distribution by Class')
axes[1].legend(['Good Credit', 'Default'])
axes[1].set_xlabel('Credit Amount')

plt.tight_layout()
plt.show()

In [ ]:
# Numeric columns correlation heatmap
numeric_df = df.select_dtypes(include=[np.number])
plt.figure(figsize=(10, 8))
sns.heatmap(numeric_df.corr(), annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, linewidths=0.5)
plt.title('Correlation Heatmap — Numeric Features')
plt.tight_layout()
plt.show()

In [ ]:
# Key feature distributions
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
features = ['credit_amount', 'duration', 'age', 'installment_rate', 'existing_credits', 'num_dependents']

for ax, feat in zip(axes.flat, features):
    for cls, color, label in [(0, '#22c55e', 'Good'), (1, '#ef4444', 'Default')]:
        df[df['target'] == cls][feat].hist(ax=ax, bins=20, alpha=0.6,
                                            color=color, label=label)
    ax.set_title(feat.replace('_', ' ').title())
    ax.legend(fontsize=8)

plt.suptitle('Feature Distributions by Credit Class', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 4. Preprocessing

In [ ]:
# Encode all categorical columns
df_enc = df.copy()
le = LabelEncoder()

for col in df_enc.select_dtypes(include='object').columns:
    df_enc[col] = le.fit_transform(df_enc[col].astype(str))

feature_cols = [c for c in df_enc.columns if c != 'target']
X = df_enc[feature_cols].values
y = df_enc['target'].values

print(f'Features: {len(feature_cols)}')
print(f'Class balance — No Default: {(y==0).sum()}, Default: {(y==1).sum()}')

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {X_train.shape[0]}, Test: {X_test.shape[0]}')

## 5. Model 1 — Logistic Regression

In [ ]:
lr = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42))
])
lr.fit(X_train, y_train)

lr_pred = lr.predict(X_test)
lr_prob = lr.predict_proba(X_test)[:, 1]

print('=== Logistic Regression ===')
print(classification_report(y_test, lr_pred, target_names=['No Default', 'Default']))
print(f'AUC-ROC: {roc_auc_score(y_test, lr_prob):.4f}')

## 6. Model 2 — Random Forest

In [ ]:
rf = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('clf', RandomForestClassifier(
        n_estimators=200, max_depth=8, class_weight='balanced',
        random_state=42, n_jobs=-1
    ))
])
rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)
rf_prob = rf.predict_proba(X_test)[:, 1]

print('=== Random Forest ===')
print(classification_report(y_test, rf_pred, target_names=['No Default', 'Default']))
print(f'AUC-ROC: {roc_auc_score(y_test, rf_prob):.4f}')

## 7. Model 3 — XGBoost

In [ ]:
scale_pos = (y_train == 0).sum() / (y_train == 1).sum()
print(f'scale_pos_weight: {scale_pos:.2f} (handles class imbalance)')

xgb = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('clf', XGBClassifier(
        n_estimators=300, max_depth=6, learning_rate=0.05,
        scale_pos_weight=scale_pos, eval_metric='auc',
        random_state=42, n_jobs=-1
    ))
])
xgb.fit(X_train, y_train)

xgb_pred = xgb.predict(X_test)
xgb_prob = xgb.predict_proba(X_test)[:, 1]

print('=== XGBoost ===')
print(classification_report(y_test, xgb_pred, target_names=['No Default', 'Default']))
print(f'AUC-ROC: {roc_auc_score(y_test, xgb_prob):.4f}')

## 8. Comparison Table + ROC Curves

In [ ]:
models = [
    ('Logistic Regression', lr_pred, lr_prob),
    ('Random Forest', rf_pred, rf_prob),
    ('XGBoost', xgb_pred, xgb_prob),
]

results = []
for name, pred, prob in models:
    results.append({
        'Model': name,
        'Accuracy':  round(accuracy_score(y_test, pred), 4),
        'Precision': round(precision_score(y_test, pred, zero_division=0), 4),
        'Recall':    round(recall_score(y_test, pred, zero_division=0), 4),
        'F1':        round(f1_score(y_test, pred, zero_division=0), 4),
        'AUC-ROC':   round(roc_auc_score(y_test, prob), 4),
    })

results_df = pd.DataFrame(results).set_index('Model')
print(results_df.to_string())
results_df.style.highlight_max(color='lightgreen', axis=0)

In [ ]:
# ROC Curves
plt.figure(figsize=(8, 6))
colors = ['#6c63ff', '#f59e0b', '#22c55e']

for (name, _, prob), color in zip(models, colors):
    fpr, tpr, _ = roc_curve(y_test, prob)
    auc = roc_auc_score(y_test, prob)
    plt.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})', color=color, linewidth=2)

plt.plot([0,1],[0,1],'--', color='gray', alpha=0.5, label='Random')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves — Model Comparison')
plt.legend()
plt.tight_layout()
plt.show()

## 9. XGBoost Feature Importance

In [ ]:
xgb_clf = xgb.named_steps['clf']
importances = xgb_clf.feature_importances_
feat_df = pd.DataFrame({'feature': feature_cols, 'importance': importances})
feat_df = feat_df.sort_values('importance', ascending=True).tail(15)

plt.figure(figsize=(9, 6))
bars = plt.barh(feat_df['feature'], feat_df['importance'], color='#6c63ff')
plt.xlabel('Importance Score')
plt.title('XGBoost Feature Importances (Top 15)')
plt.tight_layout()
plt.show()

print('\nTop 10 features:')
print(feat_df.sort_values('importance', ascending=False).head(10).to_string(index=False))

## 10. Export Best Model

In [ ]:
# XGBoost wins — save it
joblib.dump(xgb, '../backend/ml/model.pkl')
print('Model saved to ../backend/ml/model.pkl')

# Verify it loads and scores correctly
loaded = joblib.load('../backend/ml/model.pkl')
test_score = loaded.predict_proba(X_test[:1])[0][1]
print(f'Test inference — risk score: {test_score:.4f}')
print('Export verified successfully.')